# 21 — Team challengers (2026)

Teams by challenges won and lost; home vs away; by inning and count;
unused challenges at game end. Sources: `challenges_enriched`,
`challenges_remaining`, `games`, `pitches`.

In [ ]:
import os
import sys

import pandas as pd

sys.path.insert(0, "../src")
from stats_helpers import rate_table, wilson_interval

pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 40)
TABLES = "../data/tables"
REPORTS = "../reports/tables"
os.makedirs(REPORTS, exist_ok=True)

chal = pd.read_parquet(f"{TABLES}/challenges_enriched/all_seasons.parquet")
games = pd.read_parquet(f"{TABLES}/games/season=2026/part.parquet")
teams = pd.read_parquet(f"{TABLES}/teams/part.parquet")
LEAGUE_RATE = chal["is_overturned"].mean()
print(len(chal), "challenges;", len(games), "games; league rate", round(LEAGUE_RATE, 3))

In [ ]:
team_tbl = rate_table(chal, ["challenge_team_id", "challenge_team_abbr"], "is_overturned")
team_tbl = team_tbl.rename(columns={"trials": "challenges", "successes": "won"})
team_tbl["lost"] = team_tbl["challenges"] - team_tbl["won"]
team_tbl["per_game"] = (team_tbl["challenges"] / 162).round(2)
team_tbl = team_tbl.sort_values("ci_low", ascending=False).reset_index(drop=True)
team_tbl

In [ ]:
rate_table(chal, ["challenger_side"], "is_overturned")

In [ ]:
chal["inning_group"] = pd.cut(
    chal["inning"], bins=[0, 3, 6, 8, 9, 30],
    labels=["1-3", "4-6", "7-8", "9", "extra"],
)
display(rate_table(chal, ["inning_group"], "is_overturned").sort_values("inning_group"))

In [ ]:
pitches26 = pd.read_parquet(
    f"{TABLES}/pitches/season=2026/part.parquet",
    columns=["game_pk", "at_bat_index", "pitch_number", "balls_after",
             "strikes_after", "pitch_type_raw"],
)
c = chal.merge(pitches26, on=["game_pk", "at_bat_index", "pitch_number"], how="left")
# count before the pitch: undo the pitch's own effect on the final call
c["balls_pre"] = c["balls_after"] - (c["final_call_code"] == "B").astype(int)
c["strikes_pre"] = c["strikes_after"] - (c["final_call_code"] == "C").astype(int)
c["count_pre"] = c["balls_pre"].astype(str) + "-" + c["strikes_pre"].astype(str)
by_count = rate_table(c, ["count_pre"], "is_overturned", min_n=100)
display(by_count.sort_values("trials", ascending=False))

In [ ]:
g = games[games["status"].isin(["Final", "Completed Early"])].copy()
# the remaining counters are in the raw feed; rebuild from challenges_remaining's last pitch per game
rem = pd.read_parquet(f"{TABLES}/challenges_remaining/season=2026/part.parquet")
last = rem.sort_values(["game_pk", "at_bat_index", "pitch_number"]).groupby("game_pk").tail(1)
unused = pd.concat([
    last["challenges_left_home"].rename("left"),
    last["challenges_left_away"].rename("left"),
])
print("team-games:", len(unused))
print(unused.value_counts().sort_index().rename("team-games ending with N left"))
print(f"share of team-games ending with >= 1 unused: {(unused >= 1).mean():.3f}")
print(f"share ending with 2 unused (never used a failed one): {(unused == 2).mean():.3f}")

In [ ]:
chal_rem = chal.merge(
    rem, on=["game_pk", "at_bat_index", "pitch_number"], how="left"
)
chal_rem["left_before"] = chal_rem["challenges_left_home"].where(
    chal_rem["challenger_side"] == "home", chal_rem["challenges_left_away"]
)
chal_rem["late"] = chal_rem["inning"] >= 9
print("--- success by challenges left before the challenge, early vs late ---")
display(rate_table(chal_rem, ["late", "left_before"], "is_overturned")
        .sort_values(["late", "left_before"]))

In [ ]:
team_tbl.to_csv(f"{REPORTS}/team_challengers_2026.csv", index=False)
rate_table(chal, ["inning_group"], "is_overturned").to_csv(f"{REPORTS}/challenges_by_inning_2026.csv", index=False)
by_count.to_csv(f"{REPORTS}/challenges_by_count_2026.csv", index=False)
unused.value_counts().sort_index().rename_axis("left").reset_index(name="team_games").to_csv(
    f"{REPORTS}/unused_challenges_2026.csv", index=False)
print("saved")